# Constraint-Based Testing with Z3

In this exercise you use an **SMT solver** (**Z3**) to turn **logical constraints** on inputs into **concrete test data**—a small taste of *constraint-based* and *solver-backed* testing.

You start with an intuitive **Sudoku** warm-up (state the rules, let Z3 fill the blanks), then carry the same idea to the **HVAC slice** SuT **`hvac_slice_alert`** (three inputs, nonlinear intermediates, compound **`ALERT`** guard):

- Build **symbolic** versions of the same inputs and reuse **`hvac_metrics_z3`** so constraints **track the code**.
- See **`sat`** vs **`unsat`**, read **models**, and turn rationals into Python **`float`**s.
- **Target** the **`ALERT`** and **`"OK"`** regions with path conditions, then run the SuT under **`assert`**s.
- Finish with **extension TODOs** on the same formulas.

In [1]:
# If this import fails in your environment, run: pip install z3-solver
from __future__ import annotations

from z3 import Or, Real, Reals, Solver, is_rational_value, sat, simplify, unsat

print("Z3 import OK")


Z3 import OK


## Warm-up — Sudoku as a constraint problem

Before the HVAC slice, build intuition with a puzzle everyone knows: **Sudoku**.

A Sudoku is a pure **constraint-satisfaction** problem. You do **not** compute the answer step by step. Instead you **state the rules**—every row, every column, and every 3×3 box must contain the digits **1–9 exactly once**—and ask: *is there an assignment of the blanks that satisfies all the rules at once?* That is precisely the **constraint-based** mindset: describe **what must hold**, then let **Z3** find concrete values.

Below is an **almost-complete** grid with four blanks **x₁, x₂, x₃, x₄** in the top-left:

|        |        |        |   |   |   |   |   |   |
| ------ | ------ | ------ | - | - | - | - | - | - |
| 5      | 3      | 4      | 6 | 7 | 8 | 9 | 1 | 2 |
| 6      | **x₁** | **x₂** | 1 | 9 | 5 | 3 | 4 | 8 |
| 1      | **x₃** | **x₄** | 3 | 4 | 2 | 5 | 6 | 7 |
| 8      | 5      | 9      | 7 | 6 | 1 | 4 | 2 | 3 |
| 4      | 2      | 6      | 8 | 5 | 3 | 7 | 9 | 1 |
| 7      | 1      | 3      | 9 | 2 | 4 | 8 | 5 | 6 |
| 9      | 6      | 1      | 5 | 3 | 7 | 2 | 8 | 4 |
| 2      | 8      | 7      | 4 | 1 | 9 | 6 | 3 | 5 |
| 3      | 4      | 5      | 2 | 8 | 6 | 1 | 7 | 9 |

We model the four blanks as **Z3 integer variables**, keep every filled cell as a plain number, and add the three Sudoku rules as `Distinct(...)` constraints. Z3 returns the only assignment that fits.

In [26]:
# Sudoku as constraints: the four blanks are unknowns, the rules are Distinct(...) constraints.
from z3 import Distinct, Ints, IntVal, Solver, is_expr

# The four blank cells become Z3 integer variables.
x1, x2, x3, x4 = Ints("x1 x2 x3 x4")

# The board: filled cells stay as plain ints; the four blanks are Z3 variables.
# (rows top->bottom, columns left->right)
board = [
    [5, 3,  4,  6, 7, 8, 9, 1, 2],
    [6, x1, x2, 1, 9, 5, 3, 4, 8],
    [1, x3, x4, 3, 4, 2, 5, 6, 7],
    [8, 5,  9,  7, 6, 1, 4, 2, 3],
    [4, 2,  6,  8, 5, 3, 7, 9, 1],
    [7, 1,  3,  9, 2, 4, 8, 5, 6],
    [9, 6,  1,  5, 3, 7, 2, 8, 4],
    [2, 8,  7,  4, 1, 9, 6, 3, 5],
    [3, 4,  5,  2, 8, 6, 1, 7, 9],
]


def cell(x):
    """Wrap plain Python ints as Z3 terms so Distinct(...) can mix them with variables."""
    return x if is_expr(x) else IntVal(x)


s = Solver()

# Rule 0: each blank is a digit 1..9.
for v in (x1, x2, x3, x4):
    s.add(v >= 1, v <= 9)


def row_cells(r: int):
    return [cell(board[r][c]) for c in range(9)]


def col_cells(c: int):
    return [cell(board[r][c]) for r in range(9)]


def box_cells(br: int, bc: int):
    return [cell(board[br + dr][bc + dc]) for dr in range(3) for dc in range(3)]


In [27]:
rows_with_blank = [1, 2]
cols_with_blank = [1, 2]
boxes_with_blank = [(0, 0)]  # top-left 3×3 box

for r in rows_with_blank:
    s.add(Distinct(row_cells(r)))
for c in cols_with_blank:
    s.add(Distinct(col_cells(c)))
for br, bc in boxes_with_blank:
    s.add(Distinct(box_cells(br, bc)))

print("check:", s.check())  # sat -> a satisfying assignment exists
m = s.model()
print("x1 =", m[x1], " x2 =", m[x2])
print("x3 =", m[x3], " x4 =", m[x4])

check: sat
x1 = 7  x2 = 2
x3 = 9  x4 = 8


## Enjoy Sudoku!

In [28]:
# General Sudoku solver — edit `puzzle` (0 = blank) and re-run the cell.
# Unlike the minimal-rules warm-up above, this asserts the FULL Sudoku ruleset
# (every row, every column, and every box), so it reliably solves ANY valid board.
from z3 import Distinct, Int, IntVal, Solver, sat, unsat

EMPTY = 0            # blank-cell marker
MAX_EMPTY = 64       # refuse over-sparse boards (a proper Sudoku has >= 17 given clues)
TIMEOUT_MS = 10_000  # hard cap so a pathological board never hangs the notebook


def solve_sudoku(puzzle: list[list[int]]) -> list[list[int]]:
    """Solve an N×N Sudoku (N a perfect square); return the completed grid.

    Raises ValueError for a malformed / over-sparse board, RuntimeError if the
    board is unsatisfiable or is not solved within TIMEOUT_MS.
    """
    n = len(puzzle)
    box = int(n ** 0.5)
    if box * box != n or any(len(row) != n for row in puzzle):
        raise ValueError(f"board must be N×N with N a perfect square, got {n} rows")

    n_empty = sum(1 for row in puzzle for v in row if v in (0, None, EMPTY))
    if n_empty > MAX_EMPTY:
        raise ValueError(
            f"{n_empty} blanks > MAX_EMPTY={MAX_EMPTY}: too sparse to solve quickly. "
            "Fill in more givens (a real puzzle has >= 17 clues) or raise MAX_EMPTY."
        )

    s = Solver()
    s.set("timeout", TIMEOUT_MS)

    # Build the grid: givens become IntVal constants, blanks become Z3 variables in 1..n.
    board: list[list] = []
    for r in range(n):
        row = []
        for c in range(n):
            v = puzzle[r][c]
            if v in (0, None, EMPTY):
                z = Int(f"x_{r}_{c}")
                s.add(z >= 1, z <= n)
                row.append(z)
            else:
                row.append(IntVal(v))
        board.append(row)

    # Full Sudoku rules: every row, every column, and every box holds distinct digits.
    for r in range(n):
        s.add(Distinct(board[r]))
    for c in range(n):
        s.add(Distinct([board[r][c] for r in range(n)]))
    for br in range(0, n, box):
        for bc in range(0, n, box):
            s.add(Distinct([board[br + dr][bc + dc] for dr in range(box) for dc in range(box)]))

    res = s.check()
    if res == unsat:
        raise RuntimeError("unsat: this board has no valid completion (check your clues)")
    if res != sat:
        raise RuntimeError(f"solver returned '{res}' within {TIMEOUT_MS} ms — try fewer blanks")

    m = s.model()
    return [[m.eval(board[r][c]).as_long() for c in range(n)] for r in range(n)]


def print_board(grid: list[list[int]], blank_marker: int = EMPTY) -> None:
    n = len(grid)
    box = int(n ** 0.5)
    for r, row in enumerate(grid):
        if r and r % box == 0:
            print("-" * (2 * n + box - 1))
        parts = []
        for c, v in enumerate(row):
            if c and c % box == 0:
                parts.append("|")
            parts.append("." if v == blank_marker else str(v))
        print(" ".join(parts))

In [29]:
# --- change this grid freely (0 = blank) ---
puzzle = [
    [5, 3, 4, 6, 7, 8, 9, 1, 2],
    [6, 0, 0, 1, 9, 5, 3, 4, 0],
    [1, 0, 0, 3, 4, 2, 5, 6, 7],
    [8, 5, 9, 7, 6, 0, 4, 2, 3],
    [4, 2, 6, 8, 0, 3, 7, 9, 1],
    [7, 1, 3, 9, 2, 4, 0, 5, 6],
    [9, 6, 1, 5, 3, 7, 2, 8, 4],
    [2, 8, 7, 0, 1, 9, 6, 0, 5],
    [0, 0, 0, 0, 8, 6, 1, 0, 9],
]

print("blanks:", sum(1 for row in puzzle for v in row if v == EMPTY))
print("\npuzzle:")
print_board(puzzle)

solution = solve_sudoku(puzzle)
print("\nsolution (check: sat):")
print_board(solution)

blanks: 15

puzzle:
5 3 4 | 6 7 8 | 9 1 2
6 . . | 1 9 5 | 3 4 .
1 . . | 3 4 2 | 5 6 7
--------------------
8 5 9 | 7 6 . | 4 2 3
4 2 6 | 8 . 3 | 7 9 1
7 1 3 | 9 2 4 | . 5 6
--------------------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | . 1 9 | 6 . 5
. . . | . 8 6 | 1 . 9

solution (check: sat):
5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
--------------------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
--------------------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9


## From a puzzle to a program under test

Notice what just happened: we never wrote a Sudoku-solving algorithm. We **stated the rules as constraints** and Z3 produced concrete values that satisfy all of them simultaneously.

**Constraint-based testing** is the same move applied to code:

| Sudoku                          | Constraint-based testing                         |
| ------------------------------- | ------------------------------------------------ |
| blank cells `x₁…x₄`             | **test inputs** to a function                     |
| Sudoku rules (`Distinct(...)`)  | **path conditions** that mirror the code's logic  |
| Z3 model (`x₁=7, …`)            | **concrete test data** that drives a chosen branch |
| `unsat`                         | an **infeasible** path / inconsistent requirement |

The rest of this exercise applies exactly this idea to a real-looking software unit—the **HVAC slice** below.

## Software under Test (SuT) — HVAC slice with hidden nonlinear mixing

Consider a **fictitious** supervisory check on one **air-handling slice** (still safety-adjacent: bad combinations should raise **`"ALERT"`** before operators ignore a trend).

**Inputs (all `float`)**
- `supply_air_c` — supply air temperature (°C)
- `return_air_c` — return air temperature (°C)
- `flow_lps` — nominal air flow (L/s)

The code computes two aggregates—**`severity`** and **`coupling`**—through **intermediate** expressions: a **bilinear** term `supply × return`, a **quadratic** airflow penalty `flow²`, and a **trilinear-looking** gate `(supply + 2)(return − 1.5)` scaled and shifted by `flow`. The final decision is one **compound** `if`:

- **`"ALERT"`** iff `severity > 18.0` **and** `coupling > 14.5`
- otherwise **`"OK"`**

Try to “mentally pick” a witness for the **ALERT** branch: the two inequalities **propagate** through the intermediates in a way that is **not** a quick back-of-the-envelope inversion. **Z3**, however, only needs the **same** arithmetic relations over symbols.

The next cell defines the SuT and a small **`hvac_metrics_z3`** helper that mirrors the intermediates for use in constraints (keep the coefficients identical when you edit the SuT).


In [2]:
# hvac_slice_alert: the executable SuT you assert against (implementation proper).
# hvac_metrics_z3: Z3 mirror of the same arithmetic as above (constraints / path conditions).
#                  Keep coefficients and structure aligned with the SuT (drift breaks the test model).


def hvac_slice_alert(supply_air_c: float, return_air_c: float, flow_lps: float) -> str:
    """Fictitious HVAC slice: nonlinear mixing + airflow funnel into one compound guard."""
    t1 = 1.7 * supply_air_c + 0.4 * return_air_c - 0.9 * flow_lps
    t2 = (
        0.25 * supply_air_c * return_air_c
        - 0.12 * flow_lps * flow_lps
        + 3.1 * flow_lps
    )
    severity = t1 + 0.35 * t2
    coupling = (supply_air_c + 2.0) * (return_air_c - 1.5) * 0.08 + flow_lps
    if severity > 18.0 and coupling > 14.5:
        return "ALERT"
    return "OK"


def hvac_metrics_z3(s, r, f):
    """Z3 mirror of `hvac_slice_alert` intermediates (keep coefficients in sync)."""
    t1 = 1.7 * s + 0.4 * r - 0.9 * f
    t2 = 0.25 * s * r - 0.12 * f * f + 3.1 * f
    severity = t1 + 0.35 * t2
    coupling = (s + 2.0) * (r - 1.5) * 0.08 + f
    return severity, coupling


## Z3 in a nutshell — tied to the SuT’s symbols

**Z3** is an **SMT** (*Satisfiability Modulo Theories*) **solver**: you give it **first-order constraints** over symbols (here, **nonlinear** real arithmetic), and it answers **`sat`** or **`unsat`**.

- **`sat`**: there **exists** an assignment that satisfies **all** asserted constraints; Z3 returns a **model** (concrete values).
- **`unsat`**: the constraints are **jointly impossible**—useful when a path or a **requirements envelope** is **infeasible**.

The next cells use three Z3 reals that correspond to **`supply_air_c`**, **`return_air_c`**, and **`flow_lps`**, and the same **`severity` / `coupling`** pipeline as **`hvac_metrics_z3`**. Nothing here uses a separate “toy” variable set.


In [3]:
# Not using the full SuT ALERT guard yet (severity>18 ∧ coupling>14.5).
# Operating-box lower bounds: narrow the input space to a plausible region only.
# Then add severity > 10 only — loose constraints to practice sat checks and reading a model.

# Define three independent Z3 reals that correspond to the three inputs.
supply_z, return_z, flow_z = Reals("supply_z return_z flow_z")

# Use the same pipeline as the SuT to compute the intermediate expressions.
sev_z, _cross_z = hvac_metrics_z3(supply_z, return_z, flow_z)

print("--- before solve: symbols and formulas (no numbers assigned yet) ---")
print("1. supply_z:", supply_z)
print()
print("2. return_z:", return_z)
print()
print("3. flow_z:", flow_z)
print()
print("4. severity (expr over inputs):", sev_z)
print()
print("4-1. simplified severity (expr over inputs):", simplify(sev_z))
print()
print("5. coupling (expr over inputs):", _cross_z)
print()
print("5-1. simplified coupling (expr over inputs):", simplify(_cross_z))
print()

# Create a solver instance.
s_first = Solver()

# Add the operating box constraints.
s_first.add(supply_z >= 5, return_z >= 8, flow_z >= 3)

# Add the severity constraint.
s_first.add(sev_z > 10.0)

--- before solve: symbols and formulas (no numbers assigned yet) ---
1. supply_z: supply_z

2. return_z: return_z

3. flow_z: flow_z

4. severity (expr over inputs): 17/10*supply_z + 2/5*return_z - 9/10*flow_z +
7/20*
(1/4*supply_z*return_z - 3/25*flow_z*flow_z + 31/10*flow_z)

4-1. simplified severity (expr over inputs): 17/10*supply_z +
2/5*return_z +
37/200*flow_z +
7/80*return_z*supply_z +
-21/500*flow_z*flow_z

5. coupling (expr over inputs): (supply_z + 2)*(return_z - 3/2)*2/25 + flow_z

5-1. simplified coupling (expr over inputs): 2/25*(-3/2 + return_z)*(2 + supply_z) + flow_z



In [4]:
print("check:", s_first.check())
if s_first.check() == sat:
    m_first = s_first.model()
    print("model (raw):", m_first)
    # sev_z is not a declared Real — it is an expression over supply_z, return_z, flow_z.
    # Default model printing lists assignments to declared symbols only; evaluate derived exprs:
    print("derived sev_z at model:", m_first.evaluate(sev_z))
    print("derived coupling at model:", m_first.evaluate(_cross_z))
else:
    raise SystemExit("expected sat for operating box + severity > 10")

check: sat
model (raw): [supply_z = 6, flow_z = 4, return_z = 9]
derived sev_z at model: 18593/1000
derived coupling at model: 44/5


## From a Z3 model to Python `float`s

**Before** `check()`: inputs are **symbols** (`supply_z`, …) and metrics like **`sev_z`** are **formulas** built from them (see the previous cell).

**After** `sat` + `model()`: those symbols get **concrete values** (often rationals). Derived metrics are obtained with **`model.evaluate(sev_z)`**.

This cell inspects the **after-solve** values, then converts them to Python **`float`**s for **`hvac_slice_alert`** (the query above did **not** force **`ALERT`**—only **`severity > 10`** inside the box).


In [5]:
# After solve: model assigns numbers to declared symbols; evaluate() plugs them into derived exprs.
m_first = s_first.model()
raw_s, raw_r, raw_f = m_first[supply_z], m_first[return_z], m_first[flow_z]
raw_sev, raw_cross = m_first.evaluate(sev_z), m_first.evaluate(_cross_z)

print("--- after solve: raw model entries (repr) ---")
print("supply_z:", repr(raw_s), "| type:", type(raw_s).__name__)
print("return_z:", repr(raw_r), "| type:", type(raw_r).__name__)
print("flow_z:", repr(raw_f), "| type:", type(raw_f).__name__)
print("severity:", repr(raw_sev), "| type:", type(raw_sev).__name__)
print("coupling:", repr(raw_cross), "| type:", type(raw_cross).__name__)
print()

def z3_real_to_float(val) -> float:
    v = simplify(val)
    if is_rational_value(v):
        return v.numerator_as_long() / v.denominator_as_long()
    raise TypeError(f"Unsupported numeric value in model: {val!r}")


s_val = z3_real_to_float(raw_s)
r_val = z3_real_to_float(raw_r)
f_val = z3_real_to_float(raw_f)
sev_val = z3_real_to_float(raw_sev)
cross_val = z3_real_to_float(raw_cross)

print("--- converted Python floats ---")
print("supply_val:", s_val)
print("return_val:", r_val)
print("flow_val:", f_val)
print("severity_val:", sev_val)
print("coupling_val:", cross_val)
print()

--- after solve: raw model entries (repr) ---
supply_z: 6 | type: RatNumRef
return_z: 9 | type: RatNumRef
flow_z: 4 | type: RatNumRef
severity: 18593/1000 | type: RatNumRef
coupling: 44/5 | type: RatNumRef

--- converted Python floats ---
supply_val: 6.0
return_val: 9.0
flow_val: 4.0
severity_val: 18.593
coupling_val: 8.8



## When Z3 says `unsat`

If you assert **contradictory** facts about the **same** expressions (here, **`severity`** from **`hvac_metrics_z3`**), the solver returns **`unsat`**: **no** triple **`(supply, return, flow)`** satisfies them all. In testing, that pattern shows up for **infeasible paths** or **inconsistent requirements** (as in the commissioning cap later).

**Takeaway:** after **`unsat`**, there is **no** satisfying assignment, so you **cannot** obtain a witness from **`.model()`**—as the next code cell demonstrates.


In [6]:
# unsat means "no such inputs" — practice linking that to infeasible paths / inconsistent requirements.
supply_u, return_u, flow_u = Reals("supply_u return_u flow_u")
sev_u, _cross_u = hvac_metrics_z3(supply_u, return_u, flow_u)

s2 = Solver()
# The same severity scalar cannot be both very high and very low
s2.add(sev_u > 25.0, sev_u < 8.0)
print("check (expected unsat):", s2.check())
# With unsat there is no satisfying assignment, so .model() cannot yield a meaningful witness (raises).
try:
    print("s2.model() after unsat:", s2.model())
except Exception as e:
    print("s2.model() after unsat (expected failure):", e)


check (expected unsat): unsat
s2.model() after unsat (expected failure): model is not available


## Using Z3 to **target** the **`ALERT`** branch of the SuT

You already asked Z3 for a **feasible** point in the operating box with **`severity > 10`**, and for an **impossible** pair of bounds on **`severity`**. Now we target the **full branch guard** from **`hvac_slice_alert`**.

**Path condition** for **`"ALERT"`**:

- Reuse **`hvac_metrics_z3`** so **`severity`** and **`coupling`** match the code.
- Assert **`severity > 18.0`** ∧ **`coupling > 14.5`** (same strict comparisons as the `if`).

Steps:

1. Create fresh symbols for the three inputs (new names so solvers stay independent).
2. Add the same **operating box** as before if you want witnesses in a sensible range.
3. If **`sat`**, read the model, call **`hvac_slice_alert`**, and **`assert`** **`"ALERT"`**.

Again, you are not inverting the nonlinear pipeline by hand—you are **mirroring** the implementation as constraints, which is what makes **solver-backed** test generation practical.


In [7]:
def witness_alert_hvac() -> tuple[float, float, float]:
    s_, r_, f_ = Reals("supply_a return_a flow_a")
    sev, cross = hvac_metrics_z3(s_, r_, f_)
    sol = Solver()
    sol.add(s_ >= 5, r_ >= 8, f_ >= 3)

    # ALERT branch of the original SuT
    sol.add(sev > 18.0, cross > 14.5)
    
    assert sol.check() == sat
    m = sol.model()
    return (
        z3_real_to_float(m[s_]),
        z3_real_to_float(m[r_]),
        z3_real_to_float(m[f_]),
    )


alert_inputs = witness_alert_hvac()
assert hvac_slice_alert(*alert_inputs) == "ALERT"
print("ALERT witness (supply_c, return_c, flow_lps):", alert_inputs)
print("SuT ->", hvac_slice_alert(*alert_inputs))


ALERT witness (supply_c, return_c, flow_lps): (6.0, 19.0, 4.0)
SuT -> ALERT


## **`"OK"`** region and an **`unsat`** commissioning story

**`"OK"`** path: at least one of the conjuncts fails—`¬(severity > 18 ∧ coupling > 14.5)` is equivalent to `severity ≤ 18 ∨ coupling ≤ 14.5`. Z3 can still produce a witness in the operating box.

**Contradictory envelope:** imagine a **commissioning profile** that caps the **same** modeled `severity` at **12.0** while you simultaneously demand the **`ALERT`** inequalities `severity > 18` and `coupling > 14.5`. The first two cannot hold together, so Z3 returns **`unsat`**—a requirements signal, not a flaky test.


In [8]:
def witness_ok_hvac() -> tuple[float, float, float]:
    s_, r_, f_ = Reals("supply_k return_k flow_k")
    sev, cross = hvac_metrics_z3(s_, r_, f_)
    sol = Solver()
    sol.add(s_ >= 5, r_ >= 8, f_ >= 3)
    sol.add(Or(sev <= 18.0, cross <= 14.5))
    assert sol.check() == sat
    m = sol.model()
    return (
        z3_real_to_float(m[s_]),
        z3_real_to_float(m[r_]),
        z3_real_to_float(m[f_]),
    )


ok_inputs = witness_ok_hvac()
assert hvac_slice_alert(*ok_inputs) == "OK"
print("OK witness (supply_c, return_c, flow_lps):", ok_inputs)
print("SuT ->", hvac_slice_alert(*ok_inputs))

# Commissioning cap on severity contradicts ALERT's severity > 18
s3, r3, f3 = Reals("supply_c return_c flow_c")
sev3, cross3 = hvac_metrics_z3(s3, r3, f3)
sol3 = Solver()
sol3.add(sev3 <= 12.0)
sol3.add(sev3 > 18.0, cross3 > 14.5)
print("severity ≤ 12  ∧  ALERT guard:", sol3.check())
assert sol3.check() == unsat


OK witness (supply_c, return_c, flow_lps): (6.0, 9.0, 4.0)
SuT -> OK
severity ≤ 12  ∧  ALERT guard: unsat


---

## Hands-on Practice -- Pump Monitor

A different SuT: **`pump_monitor(flow_rate, pressure, viscosity)`**.

**Inputs (all `float`, operating range)**
- `flow_rate` -- volumetric flow (L/min), realistic range **5-25**
- `pressure`  -- pump outlet pressure (bar), realistic range **5-20**
- `viscosity` -- fluid dynamic viscosity (cP), realistic range **1-10**

The function computes two intermediate scalars through bilinear and quadratic mixing:

```
load = 0.3 x flow x pressure  -  0.15 x flow  +  1.2 x viscosity
wear = 0.05 x pressure^2      +  0.08 x flow x viscosity  -  0.4 x pressure
```

**Branch logic** (try to guess a `CRITICAL` triple without running the code first):

- **`"CRITICAL"`** if `load > 22.0` **and** `wear > 9.5`
- **`"WARNING"`** if `load > 22.0` **or** `wear > 9.5` (but not both)
- **`"NORMAL"`** otherwise


In [9]:
# SuT -- run this cell first, then fill in the tasks below.

def pump_monitor(flow_rate: float, pressure: float, viscosity: float) -> str:
    load = 0.3 * flow_rate * pressure - 0.15 * flow_rate + 1.2 * viscosity
    wear = 0.05 * pressure * pressure + 0.08 * flow_rate * viscosity - 0.4 * pressure
    if load > 22.0 and wear > 9.5:
        return "CRITICAL"
    if load > 22.0 or wear > 9.5:
        return "WARNING"
    return "NORMAL"


def pump_metrics_z3(f, p, v):
    """Z3 mirror of pump_monitor intermediates -- keep coefficients in sync."""
    load = 0.3 * f * p - 0.15 * f + 1.2 * v
    wear = 0.05 * p * p + 0.08 * f * v - 0.4 * p
    return load, wear


### Task 1 -- find a `"CRITICAL"` witness

Declare three Z3 reals for the inputs, call **`pump_metrics_z3`**, add the operating box and
the **CRITICAL** path condition (`load > 22.0` and `wear > 9.5`), then run the solver.
If `sat`, convert the model to floats and verify **`pump_monitor(...) == "CRITICAL"`**.


In [10]:
# Task 1 -- CRITICAL witness
# TODO: declare f1, p1, v1 = Reals("f1 p1 v1")
# TODO: build load1, wear1 with pump_metrics_z3
# TODO: add operating box (f1>=5,f1<=25, p1>=5,p1<=20, v1>=1,v1<=10) + CRITICAL guard
# TODO: check(), read model, convert to floats, assert pump_monitor(...) == "CRITICAL"


### Task 2 -- find a `"WARNING"` witness where only `load > 22` holds

Add `load > 22.0` **and** `wear <= 9.5` to stay in the WARNING-via-load branch.


In [11]:
# Task 2 -- WARNING witness (load > 22, wear <= 9.5)
# TODO: declare fresh symbols, add operating box + load > 22 + wear <= 9.5
# TODO: check(), convert, assert pump_monitor(...) == "WARNING"


### Task 3 -- find a `"WARNING"` witness where only `wear > 9.5` holds

Add `load <= 22.0` **and** `wear > 9.5` to stay in the WARNING-via-wear branch.


In [12]:
# Task 3 -- WARNING witness (load <= 22, wear > 9.5)
# TODO: declare fresh symbols, add operating box + load <= 22 + wear > 9.5
# TODO: check(), convert, assert pump_monitor(...) == "WARNING"
